# 04 — Qwen2.5-VL-3B-Instruct, tel quel

Transcription des 574 lignes de test sans aucun affinage. **CER : 38,9 %.**

**Environnement** : Google Colab avec **GPU T4**. Le modèle est chargé en `float16` (le T4 gère mal le `bfloat16`) ; il occupe environ 7,5 Go.

In [ ]:
import json, csv, unicodedata, torch
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet_vlm")
except ImportError:
    ROOT = Path("projet_vlm")

DATA = ROOT / "lignes_occitan_poly"
test = [e for e in (json.loads(l) for l in open(DATA / "lignes.jsonl", encoding="utf-8")) if e["split"] == "test"]
print(len(test), "lignes de test")

MODELE = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(MODELE)
model = AutoModelForImageTextToText.from_pretrained(MODELE, dtype=torch.float16, device_map="cuda")

In [ ]:
CONSIGNE = (
    "Transcris exactement cette ligne d'un manuscrit médiéval en occitan. "
    "Garde les abréviations telles qu'elles sont écrites, sans les développer. "
    "Réponds uniquement par la transcription."
)

def transcrire(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": CONSIGNE},
    ]}]
    entrees = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                            return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        sortie = model.generate(**entrees, max_new_tokens=100, do_sample=False)   # génération déterministe
    return processor.decode(sortie[0, entrees["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Transcription avec sauvegarde au fil de l'eau : la cellule reprend là où elle s'est arrêtée si la session est coupée.

In [ ]:
SORTIE = ROOT / "predictions_qwen_zeroshot.csv"
deja = {}
if SORTIE.exists():
    with open(SORTIE, encoding="utf-8") as f:
        deja = {r["image"]: r["pred"] for r in csv.DictReader(f)}
nouveau = not SORTIE.exists()
with open(SORTIE, "a", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    if nouveau:
        w.writerow(["image", "manuscrit", "vrai", "pred"])
    for e in tqdm(test):
        if e["image"] not in deja:
            w.writerow([e["image"], e["manuscrit"], e["text"], transcrire(Image.open(DATA / e["image"]).convert("RGB"))])
            f.flush()

In [ ]:
!pip install -q jiwer
import jiwer

def nfd(s):
    return unicodedata.normalize("NFD", s or "").strip()

with open(SORTIE, encoding="utf-8") as f:
    lignes = list(csv.DictReader(f))
vrais = [nfd(r["vrai"]) for r in lignes]
preds = [nfd(r["pred"]) for r in lignes]
print(f"CER global : {jiwer.cer(vrais, preds):.1%}")
par_ms = defaultdict(lambda: ([], []))
for r, v, p in zip(lignes, vrais, preds):
    par_ms[r["manuscrit"]][0].append(v); par_ms[r["manuscrit"]][1].append(p)
for m, (v, p) in par_ms.items():
    print(f"  {m:25} {len(v):4} lignes   CER {jiwer.cer(v, p):.1%}")